# Data Quality Anomaly Log (SETAC Europe)

**Purpose:** check every dataset for anything unusual, incorrect, incomplete or suspicious, then collect all issues in one anomaly log table.

**For each file this notebook does:** 1. load it, 2. show the shape, 3. show the columns, 4. check missing values, 5. check duplicates, 6. check data types, 7. a short markdown summary of findings.

## Step 0: Set up (finds the data folder, as per the team guide)

In [1]:
from pathlib import Path                     # Path: handles file and folder locations
import re                                    # re: used to spot ID-type column names
import warnings                              # warnings: lets us hide harmless warning messages
from difflib import get_close_matches        # finds names that are "almost" another name (catches typos)
import pandas as pd                          
import numpy as np                           

warnings.filterwarnings("ignore")            

current = Path.cwd().resolve()               # the folder this notebook is running from
project_root = next(                         # walk upwards until we find the folder that contains ".git" (the project root)
    (location for location in [current, *current.parents] if (location / ".git").exists()),
    None
)
if project_root is None:                     # stop with a clear message if we are not inside the project
    raise RuntimeError("Open the cloned SETAC-Team-12 project in VS Code, then start the notebook from that project.")

data_dir = project_root / "Fourth Rev - Data Edited"      # the agreed data folder
if not (data_dir / "Country list.xlsx").is_file():        # quick check that we are in the right place
    raise FileNotFoundError(f"Expected source file not found in: {data_dir}")

cleaned_dir = data_dir / "Cleaned"           # where our outputs are saved
cleaned_dir.mkdir(exist_ok=True)             # create it if it does not exist
print("Data folder:", data_dir)

LOW_YEAR, HIGH_YEAR = 1990, 2035             # dates outside these years will be flagged as suspicious
DATE_WORDS = ["date", "created", "modified", "booked", "graduation", "valid", "cancelled"]   # column names containing these words are treated as date columns
CONTACT_IDS = None                           # will hold the list of valid Contact IDs once Individual Contacts is loaded
anomaly_log = []                             # empty list: every problem we find is added here

Data folder: /Users/Bosola/Downloads/SETAC-Team-12/Fourth Rev - Data Edited


## Step 0b: Tools
`add_issue` writes one line into the anomaly log. `profile` does steps 2 to 6 on a table and records any problems it finds. `profile_file` loads one file and profiles it. `profile_folder` does the same for every file in a folder.

In [2]:
def add_issue(file, column, check, issue, rows, total, severity, action):
    """Add ONE line to the anomaly log."""
    anomaly_log.append({
        "file": file,                                                     # which file
        "column": column,                                                 # which column (blank if the whole file)
        "check": check,                                                   # Missing / Duplicate / Data type / Format / Link / Structure
        "issue": issue,                                                   # what is wrong, in plain English
        "rows_affected": int(rows),                                       # how many rows are affected
        "pct_of_rows": round(100 * rows / total, 1) if total else None,   # as a percentage of all rows
        "severity": severity,                                             # High / Medium / Low
        "suggested_action": action,                                       # what we think should happen
        "team_decision": ""})                                             # left blank: we willthis in later

def load(path):
    """Open one Excel or CSV file. If it is broken, record that and carry on."""
    try:
        if path.suffix.lower() == ".csv":
            return pd.read_csv(path, low_memory=False)                    # CSV file
        return pd.read_excel(path)                                        # Excel file (first sheet)
    except Exception as e:                                                # the file is missing or corrupted
        add_issue(path.name, "", "Structure", "File could not be opened: " + str(e)[:120], 0, 0, "High", "Check the file exists in the folder; ask for a fresh copy")
        print("COULD NOT OPEN:", path.name, "-", str(e)[:120])
        return None

In [3]:
def profile(df, name, needs_contact=True, show=True, unique=None):
    """Steps 2 to 6 for one table. Prints the results (if show=True) and records problems in the anomaly log."""
    n = len(df)                                                           # number of rows
    before = len(anomaly_log)                                             # how many log lines existed before this file
    key_cols = [c for c in df.columns if str(c).strip().lower() in ("contact id", "contact - id")]   # the Contact ID column, if present

    # --- the Contact ID column must exist and be spelt correctly ---
    if needs_contact and not key_cols:
        close = get_close_matches("Contact ID", [str(c) for c in df.columns], n=1, cutoff=0.7)   # look for a name that is almost "Contact ID"
        if close:
            add_issue(name, close[0], "Format", f'Misspelt header "{close[0]}" (should be "Contact ID")', n, n, "Medium", "Rename the column to Contact ID when cleaning")
            key_cols = close                                              # still use it as the Contact ID column below
        else:
            add_issue(name, "", "Structure", "No Contact ID column found, so rows cannot be linked to people", n, n, "High", "Check this is the right export")

    # --- a column that should be unique (an ID) must not repeat ---
    if unique is not None and unique in df.columns:
        repeats = int(df[unique].dropna().duplicated().sum())             # values that repeat an earlier value
        if repeats > 0:
            add_issue(name, unique, "Duplicate", f'"{unique}" should be unique but has repeated values', repeats, n, "High", "Find out why the same ID appears more than once")

    # --- 2. shape and 3. columns ---
    if show:
        print("2. SHAPE:", n, "rows x", df.shape[1], "columns")
        print("3. COLUMNS:", list(df.columns))

    # --- 4. missing values ---
    missing = df.isna().sum()
    missing = missing[missing > 0]                                        # keep only columns that have blanks
    if show:
        print("4. MISSING VALUES (only columns with blanks):")
        display(pd.DataFrame({"blank_cells": missing, "pct_blank": (100 * missing / n).round(1)}) if len(missing) else "none")
    for col, count in missing.items():
        pct = 100 * count / n
        if col in key_cols:                                               # a blank ID means the row cannot be linked to a person
            add_issue(name, col, "Missing", "Blank Contact ID: row cannot be linked to a person", count, n, "High", "Decide whether to exclude these rows")
        elif count == n:
            add_issue(name, col, "Missing", "Column is completely empty", count, n, "Medium", "Likely unused field: exclude from analysis")
        elif pct >= 5:                                                    # ignore tiny amounts of blanks
            add_issue(name, col, "Missing", "Column is blank for a large share of rows", count, n, "Medium" if pct >= 50 else "Low", "Check whether blank is expected for this field")

    # --- 5. duplicates ---
    dups = int(df.duplicated().sum())                                     # rows identical in every column to an earlier row
    if show:
        print("5. DUPLICATE ROWS (identical in every column):", dups)
    if dups > 0:
        add_issue(name, "", "Duplicate", "Rows identical in every column to an earlier row", dups, n, "Medium" if dups / n >= 0.01 else "Low", "Check whether these are genuine repeats (e.g. one row per activity) before removing")

    # --- 6. data types ---
    if show:
        print("6. DATA TYPES:")
        display(df.dtypes.astype(str).rename("type").to_frame())
    for col in df.columns:
        s = df[col].dropna()                                              # only cells that have a value
        if s.empty:
            continue
        lower = str(col).lower()
        is_text = df[col].dtype == object or pd.api.types.is_string_dtype(df[col])   # text column (old pandas says "object", new says "str")
        is_date_col = any(w in lower for w in DATE_WORDS)                 # name suggests a date
        is_id_col = re.search(r"(^|[^a-z])id($|[^a-z])", lower) is not None   # name suggests an ID
        dates = None
        if is_id_col and pd.api.types.is_float_dtype(df[col]):            # IDs should be text, not decimals
            add_issue(name, col, "Data type", "ID column stored as a decimal number", len(s), n, "Low", "Read this column as text")
        if is_text:
            if s.map(lambda v: type(v).__name__).nunique() > 1:           # e.g. some cells text, some dates
                add_issue(name, col, "Data type", "Mixed data types in one column", len(s), n, "Medium", "Convert the whole column to one type")
            if is_date_col:                                               # a date column stored as text
                parsed = pd.to_datetime(s, errors="coerce")               # try to convert; failures become blank
                bad = int(parsed.isna().sum())
                if bad < len(s):
                    add_issue(name, col, "Data type", "Dates stored as text instead of date format", len(s) - bad, n, "Medium", "Convert with pd.to_datetime()")
                if bad > 0:
                    add_issue(name, col, "Format", "Values that are not valid dates", bad, n, "High" if bad / n >= 0.05 else "Medium", "Inspect and correct the invalid entries")
                dates = parsed.dropna()
            elif not is_id_col and pd.to_numeric(s, errors="coerce").notna().mean() >= 0.9:   # 90%+ of the values are numbers
                add_issue(name, col, "Data type", "Numbers stored as text", len(s), n, "Low", "Convert the column to numbers")
        elif pd.api.types.is_datetime64_any_dtype(df[col]):               # already a proper date column
            dates = s
        if dates is not None and len(dates) > 0:
            outside = int(((dates.dt.year < LOW_YEAR) | (dates.dt.year > HIGH_YEAR)).sum())   # impossible-looking years
            if outside > 0:
                add_issue(name, col, "Format", f"Dates outside {LOW_YEAR}-{HIGH_YEAR} (earliest {dates.min().date()}, latest {dates.max().date()})", outside, n, "Medium", "Check whether these are errors or placeholder dates")
            top = dates.dt.normalize().value_counts()                     # how often each day appears
            if top.iloc[0] >= 50 and top.iloc[0] / len(dates) >= 0.10:    # one day makes up 10%+ of the values
                add_issue(name, col, "Format", f"One date ({top.index[0].date()}) repeats {top.iloc[0]} times: possible default or bulk-import date", int(top.iloc[0]), n, "Medium", "Do not treat as a genuine date in analysis")

    # --- do the Contact IDs exist in Individual Contacts? ---
    if key_cols and CONTACT_IDS is not None:
        ids = df[key_cols[0]].dropna().astype(str).str.strip().str.upper()
        unmatched = ids[~ids.isin(CONTACT_IDS)]
        if len(unmatched) > 0:
            add_issue(name, key_cols[0], "Link", f"{unmatched.nunique()} distinct Contact IDs do not exist in Individual Contacts", len(unmatched), n, "Medium", "Exclude or investigate these IDs before linking")

    return {"file": name, "rows": n, "columns": df.shape[1],
            "blank_contact_ids": int(df[key_cols[0]].isna().sum()) if key_cols else None,
            "duplicate_rows": dups, "issues_logged": len(anomaly_log) - before}

def profile_file(path, needs_contact=True, unique=None):
    """Step 1 (load) plus steps 2 to 6, for one file."""
    print("=" * 70, "\n", path.name, "\n", "=" * 70)
    df = load(path)                                                       # 1. LOAD
    if df is None:
        return None
    print("1. LOADED OK")
    profile(df, path.name, needs_contact, unique=unique)
    return df

def profile_folder(folder, needs_contact=True, names=None):
    """Same checks for every file in a folder; shows one summary table instead of 20+ long printouts."""
    if not folder.is_dir():
        print("FOLDER NOT FOUND:", folder)
        return None
    rows = []
    for path in sorted(folder.iterdir()):
        if not path.is_file() or path.suffix.lower() not in (".xlsx", ".xls", ".csv") or path.name.startswith(("~$", ".")):
            continue                                                      # skip folders, other file types and temp files
        if names is not None and path.name not in names:
            continue
        df = load(path)
        rows.append(profile(df, path.name, needs_contact, show=False) if df is not None else {"file": path.name, "rows": 0})
    summary = pd.DataFrame(rows)
    display(summary)
    return summary

# Section 1: Core files
Individual Contacts is checked first because its Contact IDs are used to test the links in every other file.

## 1.1 Individual Contacts

In [4]:
contacts = profile_file(data_dir / "Individual Contacts.xlsx", unique="Contact ID")            # load the Individual Contacts file and check it for problems
CONTACT_IDS = set(contacts["Contact ID"].dropna().astype(str).str.strip().str.upper())   # remember all valid Contact IDs for later link checks

 Individual Contacts.xlsx 
1. LOADED OK
2. SHAPE: 28922 rows x 34 columns
3. COLUMNS: ['Contact ID', 'SETAC Id', 'Website user ID', 'Job title / enrolled degree', 'Primary organisation', 'Contact type', 'Contact status', 'Default address - Country', 'Allow email', 'Opt out of all emails', 'In suppression list?', 'Is web user', 'Graduation date', 'Sector', 'Loyalty badge', 'Retired or currently not working', 'Pronouns', 'Topics interested in', 'Opted in to be a mentor', 'Include in directory', 'Show profile picture', 'Show social channels', 'Show sector', 'Show job title', 'Show directory profile biography', 'Show pronouns', 'Show areas of interest', 'Show country', 'Show email address', 'Show university / organisation', 'Last in sync date', 'Last in sync job ID', 'Date created', 'Date last modified']
4. MISSING VALUES (only columns with blanks):


,blank_cells,pct_blank
Website user ID,100,0.3
Job title / enrolled degree,8869,30.7
Primary organisation,3369,11.6
Default address - Country,1995,6.9
Allow email,2736,9.5
Graduation date,512,1.8
Sector,8316,28.8
Loyalty badge,27232,94.2
Retired or currently not working,910,3.1
Pronouns,11136,38.5


5. DUPLICATE ROWS (identical in every column): 0
6. DATA TYPES:


,type
Contact ID,str
SETAC Id,int64
Website user ID,str
Job title / enrolled degree,str
Primary organisation,str
Contact type,str
Contact status,str
Default address - Country,str
Allow email,float64
Opt out of all emails,bool


## 1.2 Subscriptions Europe

In [5]:
subs_eu = profile_file(data_dir / "Subscriptions Europe.xlsx", unique="ID")

 Subscriptions Europe.xlsx 
1. LOADED OK
2. SHAPE: 16234 rows x 47 columns
3. COLUMNS: ['Contact - ID', 'ID', 'Contact - SETAC Id', 'Contact - Web user', 'Contact - Contact type', 'Contact - Contact status', 'Status', 'Membership number', 'Product/Package', 'Product/Service', 'Renewal order is completed', 'Pause to', 'Class', 'Renewal switch to grade', 'Cancelled at', 'Renewal notification email sent at', 'Current grade', 'Renewal grade', 'Current price', 'Currency', 'Renewal switch to package', 'Duration', 'Signup order is completed', 'Subscription plan', 'Package', 'Default discount code', 'Grade', 'Is fixed package?', 'Renewal payment in progress', 'Sub status', 'Renewal approval manually', 'Recurring auth gateway config friendly name', 'Manual old price', 'Pricing method', 'Manual initial payment', 'Signup payment in progress', 'Renewal date', 'Start date', 'Renewal status', 'Is renewal switch to plan?', 'Renewal switch failed', 'Pause from', 'Auto renewal notification email sent a

,blank_cells,pct_blank
Contact - ID,2238,13.8
Contact - SETAC Id,2238,13.8
Contact - Web user,2247,13.8
Contact - Contact type,2238,13.8
Contact - Contact status,2238,13.8
Membership number,3,0.0
Pause to,16234,100.0
Renewal switch to grade,15629,96.3
Cancelled at,15182,93.5
Renewal notification email sent at,3459,21.3


5. DUPLICATE ROWS (identical in every column): 0
6. DATA TYPES:


,type
Contact - ID,str
ID,str
Contact - SETAC Id,float64
Contact - Web user,str
Contact - Contact type,str
Contact - Contact status,str
Status,str
Membership number,str
Product/Package,str
Product/Service,str


## 1.3 Country list

In [6]:
countries = profile_file(data_dir / "Country list.xlsx", needs_contact=False, unique="Country")

 Country list.xlsx 
1. LOADED OK
2. SHAPE: 243 rows x 5 columns
3. COLUMNS: ['ID', 'Country', 'Geo unit', 'Label', 'Income level']
4. MISSING VALUES (only columns with blanks):


'none'

5. DUPLICATE ROWS (identical in every column): 0
6. DATA TYPES:


,type
ID,str
Country,str
Geo unit,str
Label,str
Income level,str


# Section 2: Other single files

## 2.1 Subscriptions All

In [7]:
subs_all = profile_file(data_dir / "Subscriptions All.xlsx", unique="ID")

 Subscriptions All.xlsx 
1. LOADED OK
2. SHAPE: 40834 rows x 34 columns
3. COLUMNS: ['Contact - ID', 'Contact - SETAC Id', 'Contact - Contact type', 'Contact - Contact status', 'ID', 'Status', 'Membership number', 'Product/Package', 'Product/Service', 'Class', 'Renewal switch to grade', 'Cancelled at', 'Current grade', 'Renewal grade', 'Current price', 'Currency', 'Renewal switch to package', 'Duration', 'Signup order is completed', 'Subscription plan', 'Package', 'Default discount code', 'Grade', 'Is fixed package?', 'Renewal payment in progress', 'Sub status', 'Pricing method', 'Signup payment in progress', 'Renewal date', 'Start date', 'Is renewal switch to plan?', 'Renewal switch failed', 'Date created', 'Date last modified']
4. MISSING VALUES (only columns with blanks):


,blank_cells,pct_blank
Contact - ID,5056,12.4
Contact - SETAC Id,5056,12.4
Contact - Contact type,5056,12.4
Contact - Contact status,5056,12.4
Membership number,30,0.1
Renewal switch to grade,39237,96.1
Cancelled at,38239,93.6
Renewal grade,39237,96.1
Currency,32,0.1
Renewal switch to package,39237,96.1


5. DUPLICATE ROWS (identical in every column): 0
6. DATA TYPES:


,type
Contact - ID,str
Contact - SETAC Id,float64
Contact - Contact type,str
Contact - Contact status,str
ID,str
Status,str
Membership number,str
Product/Package,str
Product/Service,str
Class,str


## 2.2 Organisation list

In [8]:
orgs = profile_file(data_dir / "Organisation list.xlsx", needs_contact=False)

 Organisation list.xlsx 
1. LOADED OK
2. SHAPE: 5935 rows x 10 columns
3. COLUMNS: ['ID', 'Status', 'Default address - Country', 'Organisation name', 'Legal name', 'Website', 'address_country', 'SETAC Id', 'Date created', 'Date last modified']
4. MISSING VALUES (only columns with blanks):


,blank_cells,pct_blank
Status,1,0.0
Default address - Country,417,7.0
Website,867,14.6
address_country,418,7.0


5. DUPLICATE ROWS (identical in every column): 0
6. DATA TYPES:


,type
ID,str
Status,str
Default address - Country,str
Organisation name,str
Legal name,str
Website,str
address_country,str
SETAC Id,int64
Date created,datetime64[us]
Date last modified,datetime64[us]


## 2.3 Topics

In [9]:
topics = profile_file(data_dir / "Newsletters and Website Resources" / "Topics.xlsx", needs_contact=False)

 Topics.xlsx 
1. LOADED OK
2. SHAPE: 29 rows x 6 columns
3. COLUMNS: ['Topic ID', 'Topic', 'No. of contacts', 'Slug', 'Date created', 'Date last modified']
4. MISSING VALUES (only columns with blanks):


'none'

5. DUPLICATE ROWS (identical in every column): 0
6. DATA TYPES:


,type
Topic ID,str
Topic,str
No. of contacts,int64
Slug,str
Date created,datetime64[us]
Date last modified,datetime64[us]


## 2.4 Website Resources

In [10]:
website = profile_file(data_dir / "Newsletters and Website Resources" / "Website Resources.xlsx", needs_contact=False)

 Website Resources.xlsx 
1. LOADED OK
2. SHAPE: 803 rows x 15 columns
3. COLUMNS: ['ID', 'Information type', 'Type', 'Require full login', 'Access restrictions', 'Resource', 'Title', 'Full description', 'Visit count', 'resource_topics_id', 'Resource category', 'Region', 'Publish date', 'Date created', 'Date last modified']
4. MISSING VALUES (only columns with blanks):


,blank_cells,pct_blank
Information type,15,1.9
Full description,91,11.3
resource_topics_id,644,80.2
Resource category,222,27.6
Region,402,50.1


5. DUPLICATE ROWS (identical in every column): 0
6. DATA TYPES:


,type
ID,str
Information type,str
Type,str
Require full login,bool
Access restrictions,str
Resource,str
Title,str
Full description,str
Visit count,int64
resource_topics_id,str


# Section 3: Folders with many files
These cells run the same checks on every file in a folder and show one summary table per folder. Detailed problems go straight into the anomaly log.

## 3.1 In-person events and Annual Meetings (8 files)

In [11]:
events_summary = profile_folder(data_dir / "Events" / "In Person Events")

,file,rows,columns,blank_contact_ids,duplicate_rows,issues_logged
0,setac_13th_young_environmental_scientists_meet...,118,17,0,0,0
1,setac_europe_26th_lca_symposium_attendees.xlsx,514,19,1,0,2
2,setac_europe_27th_lca_symposium_attendees.xlsx,530,21,0,0,1
3,setac_europe_32nd_annual_meeting_attendees.xlsx,2289,11,532,0,5
4,setac_europe_33rd_annual_meeting_attendees.xlsx,2607,11,126,0,5
5,setac_europe_34th_annual_meeting_attendees.xlsx,3037,61,39,0,4
6,setac_europe_35th_annual_meeting_attendees.xls,2907,41,42,0,4
7,setac_europe_36th_annual_meeting_attendees.xlsx,2687,60,27,0,4


## 3.2 Webinars and SETAC Cafes (20 files)

In [12]:
webinars_summary = profile_folder(data_dir / "Events" / "Webinars")

,file,rows,columns,blank_contact_ids,duplicate_rows,issues_logged
0,Webinar_ agricultural_biologics_unpacked_scien...,144,13,0,0,1
1,Webinar_ criteria_for_reporting_and_evaluating...,180,13,0,0,1
2,Webinar_ defining_safer_drawing_a_line_in_the_...,141,13,0,0,2
3,Webinar_ driving_safer_chemicals_through_alter...,128,13,0,0,1
4,Webinar_ eco_transcriptomics_data_usability_re...,89,13,0,0,0
5,Webinar_ ecosystem_salinization_challenges_and...,126,13,0,0,1
6,Webinar_ freshwater_mussel_biology_and_laborat...,121,13,1,0,2
7,Webinar_ how_to_use_creed_attendees.xlsx,77,13,0,0,1
8,Webinar_ immunotoxicology_for_ecotoxicologists...,271,15,1,0,5
9,Webinar_ incorporating_climate_change_model_pr...,119,13,0,0,1


## 3.3 Group catalogues (3 files: not membership lists)

In [13]:
catalogues_summary = profile_folder(data_dir / "Groups", needs_contact=False)

,file,rows,columns,blank_contact_ids,duplicate_rows,issues_logged
0,Affinity Groups.xlsx,15,35,None,0,5
1,Interest Groups.xlsx,35,33,None,0,4
2,Regional Branches.xlsx,4,33,None,0,1


## 3.4 Affinity Group memberships (10 files)

In [14]:
affinity_summary = profile_folder(data_dir / "Groups" / "Affinity Groups")

,file,rows,columns,blank_contact_ids,duplicate_rows,issues_logged
0,Africa Students.xlsx,16,14,0,0,4
1,ET&C Editors.xlsx,66,14,0,0,6
2,Europe Students.xlsx,59,14,0,0,4
3,IEAM Editors.xlsx,37,14,0,0,4
4,North America Senior Resource Group.xlsx,32,14,0,0,4
5,North America Students.xlsx,51,14,0,0,4
6,North America Welcoming Inclusion.xlsx,41,14,0,0,4
7,North America Women Chemists in SETAC.xlsx,33,14,0,0,4
8,Panel on Chemicals Management.xlsx,43,14,0,0,5
9,Science Policy Advocacy – North America.xlsx,43,14,0,0,4


## 3.5 Interest Group memberships (30 files)

In [15]:
interest_summary = profile_folder(data_dir / "Groups" / "Interest Groups")

,file,rows,columns,blank_contact_ids,duplicate_rows,issues_logged
0,Advancement and Application of Alternatives As...,268,14,0,0,5
1,Animal Alternatives.xlsx,267,15,0,0,5
2,Aquatic Toxicity Testing.xlsx,588,15,0,0,6
3,Bioaccumulation Science.xlsx,442,16,0,0,6
4,Biodiversity Assessment and Conservation.xlsx,364,13,0,0,3
5,Biologics in Agriculture.xlsx,226,14,0,0,5
6,Chemistry.xlsx,638,14,0,0,4
7,Ecological Risk Assessment.xlsx,830,14,0,0,7
8,Ecotoxicology of Amphibians and Reptiles.xlsx,138,14,0,0,4
9,Effect Modeling - Europe.xlsx,264,14,0,0,6


## 3.6 Regional Branch memberships (3 files)

In [16]:
regional_summary = profile_folder(data_dir / "Groups" / "Regional Branches")

,file,rows,columns,blank_contact_ids,duplicate_rows,issues_logged
0,Europe German Language Branch.xlsx,1617,21,0,0,12
1,Europe Italian Language Branch.xlsx,701,20,0,0,8
2,Europe United Kingdom Branch.xlsx,2283,14,0,0,5


## 3.7 Newsletter campaign overviews (2 files)

In [17]:
overview_summary = profile_folder(data_dir / "Newsletters and Website Resources", needs_contact=False,
                                  names=["Overview SETAC Globe.xlsx", "Overview SETAC News.xlsx"])

,file,rows,columns,blank_contact_ids,duplicate_rows,issues_logged
0,Overview SETAC Globe.xlsx,11,23,None,0,1
1,Overview SETAC News.xlsx,13,23,None,0,0


## 3.8 Newsletter sent logs (23 files)

In [18]:
newsletter_summary = profile_folder(data_dir / "Newsletters and Website Resources" / "SETAC News and Globe Sent Logs")

,file,rows,columns,blank_contact_ids,duplicate_rows,issues_logged
0,SETAC Globe 10 September 2026.csv,20696,19,15,6,3
1,SETAC Globe 13 August 2026.csv,21721,19,510,459,3
2,SETAC Globe 16 July 2026.csv,21602,19,629,568,3
3,SETAC Globe 18 December 2025.csv,21617,19,1249,1166,3
4,SETAC Globe 18 June 2026.csv,14566,19,445,405,3
5,SETAC Globe 2 October 2025.csv,21388,19,1489,1415,4
6,SETAC Globe 23 April 2026.csv,21725,19,874,796,3
7,SETAC Globe 26 February 2026.csv,21348,19,1058,974,3
8,SETAC Globe 26 March 2026.csv,21488,19,930,850,3
9,SETAC Globe 29 January 2026.csv,21862,19,1137,1039,3


# Section 4: Issues the code cannot detect

In [19]:
manual = [
    ("(project-level)", "Loyalty badge", "Definition", "Two conflicting loyalty definitions: more than 12 months of full membership ending August 2026 vs three consecutive years ending before April 2026", "Ask SETAC which definition applies"),
    ("(project-level)", "Fees", "Definition", "Guide's fee matrix shows discounted prices at 25% of standard, but the text says 50% discount", "Ask SETAC which is correct before using price fields"),
]
for file, col, check, issue, action in manual:
    add_issue(file, col, check, issue, 0, 0, "Medium", action)             # added to the same log

# Section 5: The anomaly log

In [20]:
log = pd.DataFrame(anomaly_log)                                            # turn the list of problems into a table
order = {"High": 0, "Medium": 1, "Low": 2}                                 # so the worst problems come first
log["rank"] = log["severity"].map(order)
log = log.sort_values(["rank", "file", "check"]).drop(columns="rank").reset_index(drop=True)
log.insert(0, "log_id", range(1, len(log) + 1))                            # number every issue 1, 2, 3 ...

print("Total issues logged:", len(log))
display(log.groupby(["check", "severity"]).size().unstack(fill_value=0))   # how many issues of each type and severity
display(log[log["severity"] == "High"])                                    # the serious ones

Total issues logged: 444


severity,High,Low,Medium
check,,,
Data type,0,2,0
Definition,0,0,2
Duplicate,0,2,20
Format,1,0,56
Link,0,0,17
Missing,35,69,240


,log_id,file,column,check,issue,rows_affected,pct_of_rows,severity,suggested_action,team_decision
0,1,SETAC Globe 10 September 2026.csv,Contact ID,Missing,Blank Contact ID: row cannot be linked to a pe...,15,0.1,High,Decide whether to exclude these rows,
1,2,SETAC Globe 13 August 2026.csv,Contact ID,Missing,Blank Contact ID: row cannot be linked to a pe...,510,2.3,High,Decide whether to exclude these rows,
2,3,SETAC Globe 16 July 2026.csv,Contact ID,Missing,Blank Contact ID: row cannot be linked to a pe...,629,2.9,High,Decide whether to exclude these rows,
3,4,SETAC Globe 18 December 2025.csv,Contact ID,Missing,Blank Contact ID: row cannot be linked to a pe...,1249,5.8,High,Decide whether to exclude these rows,
4,5,SETAC Globe 18 June 2026.csv,Contact ID,Missing,Blank Contact ID: row cannot be linked to a pe...,445,3.1,High,Decide whether to exclude these rows,
5,6,SETAC Globe 2 October 2025.csv,Comtact ID,Missing,Blank Contact ID: row cannot be linked to a pe...,1489,7.0,High,Decide whether to exclude these rows,
6,7,SETAC Globe 23 April 2026.csv,Contact ID,Missing,Blank Contact ID: row cannot be linked to a pe...,874,4.0,High,Decide whether to exclude these rows,
7,8,SETAC Globe 26 February 2026.csv,Contact ID,Missing,Blank Contact ID: row cannot be linked to a pe...,1058,5.0,High,Decide whether to exclude these rows,
8,9,SETAC Globe 26 March 2026.csv,Contact ID,Missing,Blank Contact ID: row cannot be linked to a pe...,930,4.3,High,Decide whether to exclude these rows,
9,10,SETAC Globe 29 January 2026.csv,Contact ID,Missing,Blank Contact ID: row cannot be linked to a pe...,1137,5.2,High,Decide whether to exclude these rows,


In [21]:
# Save the full log. 
log.to_excel(cleaned_dir / "data_quality_anomaly_log.xlsx", index=False)
log.to_csv(cleaned_dir / "data_quality_anomaly_log.csv", index=False)
print("Saved to:", cleaned_dir)

Saved to: /Users/Bosola/Downloads/SETAC-Team-12/Fourth Rev - Data Edited/Cleaned
